# Encoding the Chategorical data

In [1]:
import pandas as pd
import numpy as np 

In [2]:
df=pd.read_csv("covid_toy.csv")

In [3]:
df.sample(5)

,age,gender,fever,cough,city,has_covid
45,72,Male,99.0,Mild,Bangalore,No
40,49,Female,102.0,Mild,Delhi,No
66,51,Male,104.0,Mild,Kolkata,No
76,80,Male,100.0,Mild,Bangalore,Yes
46,19,Female,101.0,Mild,Mumbai,No


## Normal method without column transformer

In [4]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

In [5]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(df.drop(columns=['has_covid']),df['has_covid'],test_size=0.2)

In [6]:
# removing the missing values and replacing it with the avg of it
si=SimpleImputer()
x_train_fever=si.fit_transform(x_train[['fever']])
x_test_fever=si.fit_transform(x_test[['fever']])

In [7]:
oe=OrdinalEncoder(categories=[['Mild','Strong']])
x_train_cough=oe.fit_transform(x_train[['cough']])
x_test_cough=oe.fit_transform(x_test[['cough']])

In [8]:
ohe=OneHotEncoder(drop='first',sparse_output=False)
x_train_gender_city=ohe.fit_transform(x_train[['gender','city']])
x_test_gender_city=ohe.fit_transform(x_test[['gender','city']])

In [9]:
x_train_age=x_train.drop(columns=['fever','cough','gender','city']).values
x_test_age=x_test.drop(columns=['fever','cough','gender','city']).values

In [10]:
x_train_transformed=np.concatenate((x_train_age,x_train_fever,x_train_cough,x_train_gender_city),axis=1)

In [11]:
x_train_transformed.shape

(80, 7)

## Now with the help of column transformer


In [19]:
from sklearn.compose import ColumnTransformer

transformer=ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
],remainder='passthrough')

In [22]:
transformer.fit_transform(x_train)

array([[102.        ,   1.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  82.        ],
       [104.        ,   0.        ,   1.        ,   0.        ,
          1.        ,   0.        ,  16.        ],
       [100.        ,   1.        ,   0.        ,   0.        ,
          0.        ,   0.        ,  19.        ],
       [100.88888889,   1.        ,   1.        ,   0.        ,
          1.        ,   0.        ,  79.        ],
       [101.        ,   0.        ,   1.        ,   1.        ,
          0.        ,   0.        ,  42.        ],
       [ 99.        ,   0.        ,   1.        ,   0.        ,
          0.        ,   0.        ,  72.        ],
       [ 98.        ,   0.        ,   1.        ,   0.        ,
          0.        ,   0.        ,  73.        ],
       [100.        ,   1.        ,   0.        ,   0.        ,
          1.        ,   0.        ,  13.        ],
       [ 98.        ,   0.        ,   0.        ,   0.        ,
          1.    

In [24]:
transformer.transform(x_test).shape

(20, 7)